# Inspecting GENIE cross section

Input. rootracker tree, converted from GHEP file with gntpc rootracker

First, select numu CCDIS interactions into tungsten

In [1]:
TString prepath("/eos/user/a/aiuliano/public/sims_FairShip/GenieEvents_SHIP/GenieEvents_2026_03/2026_03_16_1year_allflavours/");

TFile *gstfile = TFile::Open((prepath+TString("nu_1year_fluxhanae34.gst.root")).Data());
TFile *rootrackerfile = TFile::Open((prepath+TString("nu_1year_fluxhanae34.rootracker.root")).Data());

TTree *gsttree = (TTree*) gstfile->Get("gst");
TTree *rootrackertree = (TTree*) rootrackerfile->Get("gRooTracker");

gsttree->AddFriend(rootrackertree);

In [2]:
ROOT::RDataFrame df(*gsttree);

In [3]:
df.GetColumnNames()

(ROOT::RDF::ColumnNames_t) { "A", "DXSec", "Ef", "Ei", "El", "En", "Ev", "EvRF", "KPS", "Q2", "Q2s", "W", "Ws", "XSec", "Z", "amnugamma", "calresp0", "cc", "charm", "coh", "cthf", "cthl", "dfr", "dis", "em", "fspl", "gRooTracker.EvtCode", "gRooTracker.EvtCode.TObject", "gRooTracker.EvtCode.fString", "gRooTracker.EvtDXSec", "gRooTracker.EvtFlags", "gRooTracker.EvtFlags.TObject", "gRooTracker.EvtFlags.fAllBits", "gRooTracker.EvtFlags.fNbits", "gRooTracker.EvtFlags.fNbytes", "gRooTracker.EvtKPS", "gRooTracker.EvtNum", "gRooTracker.EvtProb", "gRooTracker.EvtVtx", "gRooTracker.EvtWght", "gRooTracker.EvtXSec", "gRooTracker.StdHepFd", "gRooTracker.StdHepFm", "gRooTracker.StdHepLd", "gRooTracker.StdHepLm", "gRooTracker.StdHepN", "gRooTracker.StdHepP4", "gRooTracker.StdHepPdg", "gRooTracker.StdHepPolz", "gRooTracker.StdHepRescat", "gRooTracker.StdHepStatus", "gRooTracker.StdHepX4", "gRooTracker.TObject", "gRooTracker.fAllBits", "gRooTracker.fNbits", "gRooTracker.fNbytes", "gRooTracker.fString",

In [4]:
auto df2 = df.Define("xsecN_overE","gRooTracker.EvtXSec/(Ev*184)");

In [5]:
//filter per neutrino type
auto df_numu = df2.Filter("neu==14");
//target must be tungsten
auto df_numu_W = df_numu.Filter("tgt==1000741840");
// interaction type CCDIS
auto df_numuccdis_W = df_numu_W.Filter("cc").Filter("dis");

In [6]:
const int nbinsE = 400;
const double minE = 0.;
const double maxE = 400.;

const int nbinsxsec = 1000;
const double minxsec = 0.;
const double maxxsec = 1.;

In [7]:
auto hsigmaE = df_numuccdis_W.Histo2D({"hsigmaE","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,1},"Ev","xsecN_overE");

In [8]:
auto hnuE = df2.Histo1D("Ev");

In [9]:
TCanvas * c = new TCanvas();
hnuE->Draw();
c->Draw();

In [10]:
TCanvas * c2 = new TCanvas();
hsigmaE->Draw();
c2->Draw();

### Splitting for target quark

There are multiple curves, they can be inspected by checking the target nucleon

In [11]:
auto df_neutron = df_numuccdis_W.Filter("hitnuc==2112");
auto df_neutron_valence = df_neutron.Filter("!sea");
auto df_neutron_sea = df_neutron.Filter("sea");

auto df_proton = df_numuccdis_W.Filter("hitnuc==2212");
auto df_proton_valence = df_proton.Filter("!sea");
auto df_proton_sea = df_proton.Filter("sea");

In [12]:
auto df_neutron_valence_up = df_neutron_valence.Filter("TMath::Abs(hitqrk) == 2");
auto df_neutron_valence_down = df_neutron_valence.Filter("TMath::Abs(hitqrk) == 1");

In [13]:
auto df_proton_valence_up = df_proton_valence.Filter("TMath::Abs(hitqrk) == 2");
auto df_proton_valence_down = df_proton_valence.Filter("TMath::Abs(hitqrk) == 1");

In [14]:
//get sub quarks from the sea
auto df_neutron_sea_up = df_neutron_sea.Filter("TMath::Abs(hitqrk) == 2");
auto df_neutron_sea_down = df_neutron_sea.Filter("TMath::Abs(hitqrk)== 1");
auto df_neutron_sea_strange = df_neutron_sea.Filter("TMath::Abs(hitqrk) == 3");
auto df_neutron_sea_charm = df_neutron_sea.Filter("TMath::Abs(hitqrk) == 4");
auto df_neutron_sea_bottom = df_neutron_sea.Filter("TMath::Abs(hitqrk) == 5");
auto df_neutron_sea_top = df_neutron_sea.Filter("TMath::Abs(hitqrk)== 6");

In [15]:
//get sub quarks from the sea proton
auto df_proton_sea_up = df_proton_sea.Filter("TMath::Abs(hitqrk)== 2");
auto df_proton_sea_down = df_proton_sea.Filter("TMath::Abs(hitqrk)== 1");
auto df_proton_sea_strange = df_proton_sea.Filter("TMath::Abs(hitqrk)== 3");
auto df_proton_sea_charm = df_proton_sea.Filter("TMath::Abs(hitqrk) == 4");
auto df_proton_sea_bottom = df_proton_sea.Filter("TMath::Abs(hitqrk) == 5");
auto df_proton_sea_top = df_proton_sea.Filter("TMath::Abs(hitqrk) == 6");

In [16]:
auto hsigmaE_nv = df_neutron_valence.Histo2D({"hsigmaE_nv","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,maxxsec},"Ev","xsecN_overE");
auto hsigmaE_ns = df_neutron_sea.Histo2D({"hsigmaE_ns","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,maxxsec},"Ev","xsecN_overE");
auto hsigmaE_pv = df_proton_valence.Histo2D({"hsigmaE_pv","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,maxxsec},"Ev","xsecN_overE");
auto hsigmaE_ps = df_proton_sea.Histo2D({"hsigmaE_ps","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,maxxsec},"Ev","xsecN_overE");

In [17]:
auto hsigmaE_nv_up = df_neutron_valence_up.Histo2D({"hsigmaE_nv_up","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,1},"Ev","xsecN_overE");
auto hsigmaE_nv_down = df_neutron_valence_down.Histo2D({"hsigmaE_nv_down","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,1},"Ev","xsecN_overE");

auto hsigmaE_pv_up = df_proton_valence_up.Histo2D({"hsigmaE_pv_up","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,1},"Ev","xsecN_overE");
auto hsigmaE_pv_down = df_proton_valence_down.Histo2D({"hsigmaE_pv_down","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,1},"Ev","xsecN_overE");

In [18]:
auto hsigmaE_ns_up = df_neutron_sea_up.Histo2D({"hsigmaE_ns_up","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,maxxsec},"Ev","xsecN_overE");
auto hsigmaE_ns_down = df_neutron_sea_down.Histo2D({"hsigmaE_ns_down","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,maxxsec},"Ev","xsecN_overE");
auto hsigmaE_ns_charm = df_neutron_sea_charm.Histo2D({"hsigmaE_ns_charm","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,maxxsec},"Ev","xsecN_overE");
auto hsigmaE_ns_strange = df_neutron_sea_strange.Histo2D({"hsigmaE_ns_strange","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,maxxsec},"Ev","xsecN_overE");
auto hsigmaE_ns_top = df_neutron_sea_top.Histo2D({"hsigmaE_ns_top","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,maxxsec},"Ev","xsecN_overE");
auto hsigmaE_ns_bottom = df_neutron_sea_bottom.Histo2D({"hsigmaE_ns_bottom","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,maxxsec},"Ev","xsecN_overE");

In [19]:
auto hsigmaE_ps_up = df_proton_sea_up.Histo2D({"hsigmaE_ps_up","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,maxxsec},"Ev","xsecN_overE");
auto hsigmaE_ps_down = df_proton_sea_down.Histo2D({"hsigmaE_ps_down","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,maxxsec},"Ev","xsecN_overE");
auto hsigmaE_ps_charm = df_proton_sea_charm.Histo2D({"hsigmaE_ps_charm","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,maxxsec},"Ev","xsecN_overE");
auto hsigmaE_ps_strange = df_proton_sea_strange.Histo2D({"hsigmaE_ps_strange","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,maxxsec},"Ev","xsecN_overE");
auto hsigmaE_ps_top = df_proton_sea_top.Histo2D({"hsigmaE_ps_top","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,maxxsec},"Ev","xsecN_overE");
auto hsigmaE_ps_bottom = df_proton_sea_bottom.Histo2D({"hsigmaE_ps_bottom","Cross Section numu CCDIS",nbinsE,minE,maxE,nbinsxsec,minxsec,maxxsec},"Ev","xsecN_overE");

In [20]:
TCanvas *cp_v = new TCanvas();

cp_v->Divide(2,1);
cp_v->cd(1);
hsigmaE_pv_up->Draw();
cp_v->cd(2);
hsigmaE_pv_down->Draw();
cp_v->Draw();

In [21]:
TCanvas *cn_v = new TCanvas();

cn_v->Divide(2,1);
cn_v->cd(1);
hsigmaE_nv_up->Draw();
cn_v->cd(2);
hsigmaE_nv_down->Draw();
cn_v->Draw();

In [22]:
TCanvas *cn_s = new TCanvas();
cn_s->Divide(3,2);
cn_s->cd(1);
hsigmaE_ns_up->Draw();
cn_s->cd(2);
hsigmaE_ns_down->Draw();
cn_s->cd(3);
hsigmaE_ns_charm->Draw();
cn_s->cd(4);
hsigmaE_ns_strange->Draw();
cn_s->cd(5);
hsigmaE_ns_top->Draw();
cn_s->cd(6);
hsigmaE_ns_bottom->Draw();
cn_s->Draw();

In [23]:
TCanvas *cp_s = new TCanvas();
cp_s->Divide(3,2);
cp_s->cd(1);
hsigmaE_ps_up->Draw();
cp_s->cd(2);
hsigmaE_ps_down->Draw();
cp_s->cd(3);
hsigmaE_ps_charm->Draw();
cp_s->cd(4);
hsigmaE_ps_strange->Draw();
cp_s->cd(5);
hsigmaE_ps_top->Draw();
cp_s->cd(6);
hsigmaE_ps_bottom->Draw();
cp_s->Draw();

## Trying to sum them together

In [24]:
auto prof_sigmaE_pv = df_proton_valence.Profile1D({"prof_sigmaE_pv","Cross Section numu CCDIS",nbinsE,minE,maxE,minxsec,maxxsec},"Ev","xsecN_overE");
auto prof_sigmaE_nv = df_neutron_valence.Profile1D({"prof_sigmaE_nv","Cross Section numu CCDIS",nbinsE,minE,maxE,minxsec,maxxsec},"Ev","xsecN_overE");

auto prof_sigmaE_ps_up = df_proton_sea_up.Profile1D({"prof_sigmaE_ps_up","Cross Section numu CCDIS",nbinsE,minE,maxE,minxsec,maxxsec},"Ev","xsecN_overE");
auto prof_sigmaE_ps_down = df_proton_sea_down.Profile1D({"prof_sigmaE_ps_down","Cross Section numu CCDIS",nbinsE,minE,maxE,minxsec,maxxsec},"Ev","xsecN_overE");
auto prof_sigmaE_ps_charm = df_proton_sea_charm.Profile1D({"prof_sigmaE_ps_charm","Cross Section numu CCDIS",nbinsE,minE,maxE,minxsec,maxxsec},"Ev","xsecN_overE");
auto prof_sigmaE_ps_strange = df_proton_sea_strange.Profile1D({"prof_sigmaE_ps_strange","Cross Section numu CCDIS",nbinsE,minE,maxE,minxsec,maxxsec},"Ev","xsecN_overE");
auto prof_sigmaE_ps_top = df_proton_sea_top.Profile1D({"prof_sigmaE_ps_top","Cross Section numu CCDIS",nbinsE,minE,maxE,minxsec,maxxsec},"Ev","xsecN_overE");
auto prof_sigmaE_ps_bottom = df_proton_sea_bottom.Profile1D({"prof_sigmaE_ps_down","Cross Section numu CCDIS",nbinsE,minE,maxE,minxsec,maxxsec},"Ev","xsecN_overE");

auto prof_sigmaE_ns_up = df_neutron_sea_up.Profile1D({"prof_sigmaE_ns_up","Cross Section numu CCDIS",nbinsE,minE,maxE,minxsec,maxxsec},"Ev","xsecN_overE");
auto prof_sigmaE_ns_down = df_neutron_sea_down.Profile1D({"prof_sigmaE_ns_down","Cross Section numu CCDIS",nbinsE,minE,maxE,minxsec,maxxsec},"Ev","xsecN_overE");
auto prof_sigmaE_ns_charm = df_neutron_sea_charm.Profile1D({"prof_sigmaE_ns_charm","Cross Section numu CCDIS",nbinsE,minE,maxE,minxsec,maxxsec},"Ev","xsecN_overE");
auto prof_sigmaE_ns_strange = df_neutron_sea_strange.Profile1D({"prof_sigmaE_ns_strange","Cross Section numu CCDIS",nbinsE,minE,maxE,minxsec,maxxsec},"Ev","xsecN_overE");
auto prof_sigmaE_ns_top = df_neutron_sea_top.Profile1D({"prof_sigmaE_ns_top","Cross Section numu CCDIS",nbinsE,minE,maxE,minxsec,maxxsec},"Ev","xsecN_overE");
auto prof_sigmaE_ns_bottom = df_neutron_sea_bottom.Profile1D({"prof_sigmaE_ns_down","Cross Section numu CCDIS",nbinsE,minE,maxE,minxsec,maxxsec},"Ev","xsecN_overE");


In [25]:
TCanvas *ctot = new TCanvas();
// 1. Ottieni gli oggetti TH1D dai profili
// ProjectionX() crea un nuovo istogramma con i valori medi del profilo
TH1D* h1 = prof_sigmaE_pv->ProjectionX("h1");
TH1D* h2 = prof_sigmaE_nv->ProjectionX("h2");

TH1D* h3 = prof_sigmaE_ps_up->ProjectionX("h3");
TH1D* h4 = prof_sigmaE_ps_down->ProjectionX("h4");
TH1D* h5 = prof_sigmaE_ps_charm->ProjectionX("h5");
TH1D* h6 = prof_sigmaE_ps_strange->ProjectionX("h6");
TH1D* h7 = prof_sigmaE_ps_top->ProjectionX("h7");
TH1D* h8 = prof_sigmaE_ps_bottom->ProjectionX("h8");

TH1D* h9 = prof_sigmaE_ns_up->ProjectionX("h9");
TH1D* h10 = prof_sigmaE_ns_down->ProjectionX("h10");
TH1D* h11 = prof_sigmaE_ns_charm->ProjectionX("h11");
TH1D* h12 = prof_sigmaE_ns_strange->ProjectionX("h12");
TH1D* h13 = prof_sigmaE_ns_top->ProjectionX("h13");
TH1D* h14 = prof_sigmaE_ns_bottom->ProjectionX("h14");

// 2. Crea l'istogramma somma
TH1D* h_sum = (TH1D*)h1->Clone("h_sum");
h_sum->SetTitle("Somma non pesata (y1 + y2)");
// 3. Esegui la somma aritmetica
h_sum->Add(h2); h_sum->Add(h3); h_sum->Add(h4); h_sum->Add(h5); h_sum->Add(h6); h_sum->Add(h7);
h_sum->Add(h8); h_sum->Add(h9); h_sum->Add(h10); h_sum->Add(h11); h_sum->Add(h12); h_sum->Add(h13);
h_sum->Add(h14);

h_sum->Draw();
ctot->Draw();